# Malaria Detection System

## 1. Introduction
In this project, we'll develop a Malaria Detection System using a Convolutional Neural Network (CNN) for image classification and a Graphical User Interface (GUI) for user interaction. The system will:

### Train a CNN on a dataset of blood smear images to classify them into different malaria species.
### Provide a GUI that allows users to upload images and receive real-time predictions.
This comprehensive approach ensures that the model is not only trained effectively but also easily accessible for practical use.

## 2. Imports
First, we'll import all the necessary libraries required for data processing, model building, training, evaluation, and GUI creation.

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from tensorflow.keras.models import load_model
import tkinter as tk
from tkinter import filedialog, ttk
from PIL import Image, ImageOps, ImageTk
import cv2  # Import OpenCV for broader image support
from collections import Counter


## 3. Define Dataset Directory and Classes
We'll specify the directory where our dataset is stored and define the class names corresponding to different malaria species

In [3]:
# -----------------------------
# 3. Define Dataset Directory and Classes 
# -----------------------------
dataset_dir = r"d:\ai progect\dataset\dataset"  # Update this path as needed
class_names = ['falciparum', 'malariae', 'ovale', 'vivax']
num_classes = len(class_names)


## 4. Gather File Paths and Labels
We'll traverse through each class directory to collect image file paths and their corresponding labels.

In [4]:
# -----------------------------
# 4. Gather File Paths and Labels
# -----------------------------
file_paths = []
labels = [] 

for label, class_name in enumerate(class_names):
    class_dir = os.path.join(dataset_dir, class_name)
    if not os.path.isdir(class_dir):
        print(f"Warning: Directory {class_dir} does not exist.")
        continue
    for fname in os.listdir(class_dir):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp', '.avif')):
            file_paths.append(os.path.join(class_dir, fname))
            labels.append(label)

file_paths = np.array(file_paths)
labels = np.array(labels)

print(f"Total samples: {len(file_paths)}")


Total samples: 2203


## 5. Split into Train, Validation, Test Sets
We'll split the dataset into training, validation, and testing subsets to evaluate the model's performance effectively.

In [5]:
# -----------------------------
# 5. Split into Train, Validation, Test
# -----------------------------
# First split: Train + Val and Test
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    file_paths,
    labels,
    test_size=0.10,  # 10% for test
    stratify=labels,
    random_state=42
)

# Second split: Train and Validation
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths,
    train_val_labels,
    test_size=0.10,  # 0.1111 * 0.90 ≈ 10% of total data for validation
    stratify=train_val_labels,
    random_state=42
)

print(f"Training samples: {len(train_paths)}")
print(f"Validation samples: {len(val_paths)}")
print(f"Test samples: {len(test_paths)}")


Training samples: 1761
Validation samples: 221
Test samples: 221


## 6. Create TensorFlow Datasets
We'll convert the file paths and labels into TensorFlow Dataset objects for efficient data loading and preprocessing during training.

In [6]:
# -----------------------------
# 6. Create TensorFlow Datasets
# -----------------------------
img_height = 150
img_width = 150
batch_size = 16  # Adjust based on your hardware capabilities

def process_path(file_path, label):
    image = tf.io.read_file(file_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])  # Ensure image has 3 channels
    image = tf.image.resize(image, [img_height, img_width])
    image = image / 255.0  # Normalize to [0,1]
    return image, label

def filter_invalid_images(image, label):
    # Ensure image has shape (img_height, img_width, 3)
    return tf.logical_and(
        tf.equal(tf.shape(image)[-1], 3),
        tf.logical_and(tf.shape(image)[0] > 0, tf.shape(image)[1] > 0)
    )

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.filter(filter_invalid_images)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.filter(filter_invalid_images)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.filter(filter_invalid_images)


## 7. Data Augmentation
Data augmentation artificially expands the training dataset by applying random transformations, helping the model generalize better.

python
Copy code


In [7]:
# -----------------------------
# 7. Data Augmentation
# -----------------------------
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

def augment(image, label):
    image = data_augmentation(image)
    return image, label

# Apply data augmentation only to training data
train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(buffer_size=1000)
train_ds = train_ds.batch(batch_size)
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
train_ds = train_ds.repeat()  # Repeat indefinitely for training

val_ds = val_ds.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = test_ds.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)


## 8. Calculate Class Weights
Class weights help address class imbalance by assigning higher weights to minority classes during training.

In [8]:
# -----------------------------
# 8. Calculate Class Weights
# -----------------------------
# Count labels in training set
train_labels_flat = train_labels
label_counts = Counter(train_labels_flat)
total_samples = sum(label_counts.values())

# Compute class weights
class_weights = {}
for i in range(num_classes):
    if i in label_counts:
        class_weights[i] = total_samples / (num_classes * label_counts[i])
    else:
        print(f"Warning: Class {i} not found in training data.")
        class_weights[i] = 1.0  # Default weight

print(f"Class Weights: {class_weights}")


Class Weights: {0: 0.6804482225656878, 1: 15.181034482758621, 2: 19.141304347826086, 3: 0.4145480225988701}


## 9. Define and Compile the CNN Model
We'll construct a Convolutional Neural Network (CNN) tailored for image classification tasks.

python
Copy code


In [9]:
# -----------------------------
# 9. Define and Compile the CNN Model
# -----------------------------
model = tf.keras.Sequential([
    # Block 1
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3),
                           kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),
    
    # Block 2
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu',
                           kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),
    
    # Flatten and Dense Layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu',
                          kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Learning rate schedule
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.001,  # Adjusted for simplified model
    decay_steps=1000
)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              loss='sparse_categorical_crossentropy',  # Using integer labels
              metrics=['accuracy'])

model.summary()


c:\Users\RTX\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 148, 148, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 72, 72, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 82944)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    10,616,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,637,764 (40.58 MB)

 Trainable params: 10,637,316 (40.58 MB)

 Non-trainable params: 448 (1.75 KB)

## 10. Train the Model with Early Stopping
We'll train the CNN model using the training dataset, incorporating early stopping to prevent overfitting.

In [ ]:
# -----------------------------
# 10. Train the Model with Early Stopping
# -----------------------------
model_path = 'simplified_malaria_model.h5'  # Path to save/load the model

if not os.path.exists(model_path):
    print("Training the model as no saved model found...")
    
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    epochs = 20  # Adjusted epochs for simpler model
    
    # Calculate steps_per_epoch
    steps_per_epoch = np.ceil(len(train_paths) / batch_size).astype(int)
    validation_steps = np.ceil(len(val_paths) / batch_size).astype(int)
    
    history = model.fit(
        train_ds,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_ds,
        validation_steps=validation_steps,
        class_weight=class_weights,
        callbacks=[early_stopping]
    )
    
    # Save the best model
    model.save(model_path)
    print(f"Model trained and saved at {model_path}")
else:
    print(f"Loading existing model from {model_path}...")
    model = load_model(model_path)


## 11. Evaluate on Test Set
After training, we'll evaluate the model's performance on the unseen test dataset.

In [ ]:
# -----------------------------
# 11. Evaluate on Test Set
# -----------------------------
if not os.path.exists(model_path):
    print("No model found to evaluate.")
else:
    print("Evaluating the model on the test set...")
    test_loss, test_acc = model.evaluate(test_ds, verbose=1)
    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")


## 12. Plot Training Curves
Visualizing the training and validation metrics helps in understanding the model's learning behavior and diagnosing issues like overfitting.

In [ ]:
# -----------------------------
# 12. Plot Training Curves
# -----------------------------
# Plot Training Curves (only if model was trained in this run)
if 'history' in locals():
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.show()


## 13. Confusion Matrix & Classification Report
We'll analyze the model's performance in detail by generating a confusion matrix and a classification report.

In [ ]:
# -----------------------------
# 13. Confusion Matrix & Classification Report
# -----------------------------
# Gather predictions and true labels
y_pred = []
y_true = []

for images, labels_batch in test_ds:
    preds = model.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels_batch.numpy())

y_pred = np.array(y_pred)
y_true = np.array(y_true)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))


## 14. Build GUI for Image Prediction
We'll create a Graphical User Interface (GUI) using Tkinter that allows users to upload images and receive real-time predictions.

In [ ]:
# -----------------------------
# 14. Build GUI for Image Prediction
# -----------------------------
def predict_image_gui():
    if not hasattr(predict_image_gui, 'file_path'):
        result_label.config(text="Please upload an image first.")
        return
    
    file_path = predict_image_gui.file_path
    try:
        # Preprocess the image with OpenCV
        img_cv = cv2.imread(file_path, cv2.IMREAD_UNCHANGED)  # Read image with OpenCV
        if img_cv is None:
            result_label.config(text="Error: Unable to read the image. Unsupported format.")
            return

        # Convert to RGB if not already
        if len(img_cv.shape) == 2 or img_cv.shape[2] == 4:  # Grayscale or RGBA
            img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGRA2RGB if img_cv.shape[2] == 4 else cv2.COLOR_GRAY2RGB)

        # Resize to model input size
        img_resized = cv2.resize(img_cv, (img_width, img_height))
        image_array = np.array(img_resized) / 255.0  # Normalize
        image_array = np.expand_dims(image_array, axis=0)

        # Make predictions
        predictions = model.predict(image_array)
        predicted_class = class_names[np.argmax(predictions)]
        confidence_scores = {class_names[i]: float(predictions[0][i]) for i in range(num_classes)}
        
        # Prepare result text
        result_text = f"Predicted Class: {predicted_class}\nConfidence Scores:\n"
        for cls, score in confidence_scores.items():
            result_text += f"{cls}: {score*100:.2f}%\n"
        
        result_label.config(text=result_text)
    except Exception as e:
        result_label.config(text=f"An error occurred during prediction: {e}")

def upload_image_gui():
    file_path = filedialog.askopenfilename(
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp *.gif *.webp *.avif"), ("All files", "*.*")]
    )
    if not file_path:
        result_label.config(text="No file selected")
        return
    
    try:
        # Display the uploaded image using OpenCV and convert to RGB
        img_cv = cv2.imread(file_path, cv2.IMREAD_UNCHANGED)  # Read image with OpenCV
        if img_cv is None:
            result_label.config(text="Error: Unsupported file format. Please upload a valid image.")
            return

        # Convert to RGB if not already
        if len(img_cv.shape) == 2 or img_cv.shape[2] == 4:  # Grayscale or RGBA
            img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGRA2RGB if img_cv.shape[2] == 4 else cv2.COLOR_GRAY2RGB)

        # Resize for display purposes
        img_resized = cv2.resize(img_cv, (300, 300))
        img_pil = Image.fromarray(img_resized)
        tk_img = ImageTk.PhotoImage(img_pil)
        image_label.config(image=tk_img)
        image_label.image = tk_img
        result_label.config(text="Image uploaded successfully. Click 'Predict' to proceed.")
        
        # Store the file_path for prediction
        predict_image_gui.file_path = file_path
    except Exception as e:
        result_label.config(text=f"Error: {e}")

# Initialize GUI
root = tk.Tk()
root.title("Malaria Detection Professional GUI")
root.geometry("700x800")

# Title Label
title_label = ttk.Label(root, text="Malaria Detection System", font=("Arial", 22, "bold"))
title_label.pack(pady=20)

# Image Display Area
image_label = ttk.Label(root, text="Image will appear here", anchor="center", background="#dcdcdc", width=50, relief="solid")
image_label.pack(pady=20)

# Result Box
result_label = ttk.Label(root, text="Prediction result will appear here.", font=("Arial", 14), background="#ffffff", anchor="center", width=60, relief="solid", wraplength=400, justify="center")
result_label.pack(pady=20)

# Upload Button
upload_button = ttk.Button(root, text="Upload Image", command=upload_image_gui)
upload_button.pack(pady=10)

# Predict Button
predict_button = ttk.Button(root, text="Predict", command=predict_image_gui)
predict_button.pack(pady=10)

root.mainloop()
